In [ ]:
import os

REPO = "MLOPS_group_project"
REPO_URL = "https://github.com/Tejinder482/grp_24_mlop_project.git"
BRANCH = "develop"

if os.path.isdir(REPO):
    %cd {REPO}
    !git fetch origin {BRANCH}
    !git checkout {BRANCH}
    !git pull origin {BRANCH}
else:
    !git clone -b {BRANCH} {REPO_URL}
    %cd {REPO}

!pip install -q -r requirements.txt

In [ ]:
import os

import wandb
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from kaggle_web_client import BackendError

# Add-ons → Secrets → create HF_TOKEN and WANDB_API_KEY → attach to THIS notebook
REQUIRED_SECRETS = ("HF_TOKEN", "WANDB_API_KEY")

secrets = UserSecretsClient()
for label in REQUIRED_SECRETS:
    try:
        secrets.get_secret(label)
    except BackendError as exc:
        raise RuntimeError(
            f"Missing Kaggle secret '{label}'. "
            "Open Add-ons → Secrets → add the secret → attach it to THIS notebook → Save → restart session."
        ) from exc

os.environ["WANDB_API_KEY"] = secrets.get_secret("WANDB_API_KEY")
os.environ["PLATFORM"] = "Kaggle"
login(token=secrets.get_secret("HF_TOKEN"))
wandb.login()
print("HF and W&B login successful.")

In [ ]:
!python src/preprocess.py

In [ ]:
!python src/train.py --config configs/train_v2.yaml

In [ ]:
HUB_MODEL_ID = "tejinder482/imdb-distilbert-best"
!python src/push_to_hub.py --config configs/train_v2.yaml --hub-model-id {HUB_MODEL_ID}